In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
sim.AddCompound("Methanol")
sim.AddCompound("Acetic acid")
sim.AddCompound("Carbon monoxide")
sim.AddCompound("Water")

In [5]:
feed = sim.AddFlowsheetObject('Material Stream','feed')
product = sim.AddFlowsheetObject('Material Stream','product')
energy_stream = sim.AddFlowsheetObject('Energy Stream','energy_stream')
cstr_01 = sim.AddFlowsheetObject('Continuous Stirred Tank Reactor (CSTR)','cstr_01')

In [6]:
feed = feed.GetAsObject()
product = product.GetAsObject()
energy_stream = energy_stream.GetAsObject()
cstr_01 = cstr_01.GetAsObject()

In [7]:
sim.ConnectObjects(feed.GraphicObject, cstr_01.GraphicObject, -1, -1)
sim.ConnectObjects(cstr_01.GraphicObject, product.GraphicObject, -1, -1)
sim.ConnectObjects(energy_stream.GraphicObject, cstr_01.GraphicObject, -1, -1)

In [8]:
sim.AutoLayout()

In [9]:
feed.SetOverallComposition(Array[float]([0.52, 0.0, 0.01061, 0.4694])) # Feed composition (50% CO and 50% H2O)
feed.SetTemperature(328.0) # K
feed.SetPressure(101325.0) # Pa
feed.SetMassFlow(1.0) # kg/s

'feed: mass flow set to 1 kg/s'

In [10]:
# property package
Thermo_Package_UNIQUAC = sim.CreateAndAddPropertyPackage("UNIQUAC")

In [11]:
# Creating the conversion reaction for CH4 combustion
from System.Collections.Generic import Dictionary
from System import String, Double

# ------------------------------------------------------------
# 2. Define stoichiometry and kinetic orders
# ------------------------------------------------------------
# Stoichiometry: Methanol + CO -> Acetic acid
stoich = Dictionary[String, Double]()
stoich.Add("Methanol", -1.0)
stoich.Add("Carbon monoxide", -1.0)
stoich.Add("Acetic acid", 1.0)

# Direct (forward) reaction orders
direct_orders = Dictionary[String, Double]()
direct_orders.Add("Methanol", 1.0)
direct_orders.Add("Carbon monoxide", 0.5)   # example: fractional order
direct_orders.Add("Acetic acid", 0.0)

# Reverse reaction orders (set to zero for irreversible reaction)
reverse_orders = Dictionary[String, Double]()
reverse_orders.Add("Methanol", 0.0)
reverse_orders.Add("Carbon monoxide", 0.0)
reverse_orders.Add("Acetic acid", 0.0)

# Base compound (required, usually the main reactant)
base_compound = "Methanol"

# Reaction phase (Mixture, Vapor, Liquid, etc.)
reaction_phase = "Mixture"   # both phases considered

# Basis – "Molar Concentration" means rate depends on molarity (mol/m³)
basis = "Molar Concentration"

# Units for amount (concentration) and reaction rate
amount_units = "mol/m³"      # consistent with molar concentration
rate_units   = "mol/m³.s"    # rate per unit volume per second

# Arrhenius parameters for forward reaction
Aforward = 3500000             # pre‑exponential factor (same units as rate)
Eforward = 83680            # activation energy (J/mol)

# Reverse reaction parameters (set to zero for irreversible)
Areverse = 0.0
Ereverse = 0.0

# Optional user‑defined expressions (leave empty to use Arrhenius)
Expr_forward = ""
Expr_reverse = ""

# Create the kinetic reaction
kin_reaction = sim.CreateKineticReaction(
    "MethanolCarbonylation",                # name
    "CH3OH + CO -> CH3COOH (kinetic)",      # description
    stoich,                                  # stoichiometry
    direct_orders,                           # forward orders
    reverse_orders,                          # reverse orders
    base_compound,                           # base compound
    reaction_phase,                          # phase
    basis,                                    # basis
    amount_units,                             # amount units
    rate_units,                               # rate units
    Aforward, Eforward, Areverse, Ereverse,  # Arrhenius parameters
    Expr_forward, Expr_reverse                # custom expressions
)

# Add reaction to flowsheet
sim.AddReaction(kin_reaction)

# ------------------------------------------------------------
# 3. Create a reaction set and add the kinetic reaction
# ------------------------------------------------------------
reaction_set = sim.CreateReactionSet("Kinetic Set", "Contains methanol carbonylation")
sim.AddReactionSet(reaction_set)
sim.AddReactionToSet(kin_reaction.Name, reaction_set.Name, True, 0)

# Setting reaction set to the conversion reactor
from DWSIM.UnitOperations.Reactors import OperationMode
cstr_01.ReactorOperationMode = OperationMode.Adiabatic
cstr_01.DeltaP = 0.5
cstr_01.ReactionSetID = reaction_set.Name
cstr_01.ReactionSet = reaction_set
cstr_01.m_headspace = 0.001
cstr_01.m_vol = 100
cstr_01.CatalystAmount = 1.0

In [12]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [13]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/14 CSTR Automation/14 CSTR Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)